In [0]:
print("Student Performance Data Pipeline")
print("Starting our first data engineering project!")

Student Performance Data Pipeline
Starting our first data engineering project!


In [0]:
df = spark.table("workspace.default.project_student_performance") # this is path takem from catalog -> table
display(df)

student_id,student_name,department,semester,marks,attendance_percentage,backlogs,fees_paid,placement_status
1001,Aditya Singh,CSE,3,69,78,1,Yes,Placed
1002,Riya Rana,IT,7,68,92,1,Yes,Not Placed
1003,Aadhya Bhatt,CSE,1,85,77,1,No,Not Placed
1004,Aditya Nair,CSE,4,85,82,1,Yes,Not Placed
1005,Krishna Shah,ECE,7,67,66,2,Yes,Placed
1006,Anika Parmar,IT,8,68,89,0,Yes,Not Placed
1007,Rahul Singh,Civil,3,82,100,1,Yes,Not Placed
1008,Priya Singh,CSE,6,62,89,1,Yes,Not Placed
1009,Anika Mehta,ECE,2,68,97,0,Yes,Placed
1010,Nisha Chauhan,IT,1,75,74,0,Yes,Placed


In [0]:
print("Total rows :",df.count())
print("Total coloumns : ",len(df.columns))

Total rows : 1000
Total coloumns :  9


In [0]:
df.printSchema()


root
 |-- student_id: long (nullable = true)
 |-- student_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- semester: long (nullable = true)
 |-- marks: long (nullable = true)
 |-- attendance_percentage: long (nullable = true)
 |-- backlogs: long (nullable = true)
 |-- fees_paid: string (nullable = true)
 |-- placement_status: string (nullable = true)



In [0]:
df.dropna().count()

1000

In [0]:
df.filter(df.student_name.isNull()).count()

0

In [0]:
df.groupBy("student_id").count().filter("count > 1").show()

+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+



In [0]:
df.filter(df.marks < 40).show()

+----------+---------------+----------+--------+-----+---------------------+--------+---------+----------------+
|student_id|   student_name|department|semester|marks|attendance_percentage|backlogs|fees_paid|placement_status|
+----------+---------------+----------+--------+-----+---------------------+--------+---------+----------------+
|      1262|     Ayaan Dave|        IT|       2|   34|                   88|       1|      Yes|      Not Placed|
|      1490|   Ananya Gupta|        IT|       4|   36|                   84|       2|      Yes|      Not Placed|
|      1508|     Anika Dave|        IT|       5|   39|                   78|       1|      Yes|      Not Placed|
|      1821|     Priya Shah|     Civil|       5|   36|                   74|       1|      Yes|      Not Placed|
|      1910|Reyansh Trivedi|Mechanical|       5|   36|                   74|       0|      Yes|      Not Placed|
+----------+---------------+----------+--------+-----+---------------------+--------+---------+-

In [0]:
df_clean = df.dropna() # removes rows containing missing values.
df_clean = df_clean.dropDuplicates(["student_id"])

display(df_clean)

student_id,student_name,department,semester,marks,attendance_percentage,backlogs,fees_paid,placement_status
1001,Aditya Singh,CSE,3,69,78,1,Yes,Placed
1002,Riya Rana,IT,7,68,92,1,Yes,Not Placed
1003,Aadhya Bhatt,CSE,1,85,77,1,No,Not Placed
1004,Aditya Nair,CSE,4,85,82,1,Yes,Not Placed
1005,Krishna Shah,ECE,7,67,66,2,Yes,Placed
1006,Anika Parmar,IT,8,68,89,0,Yes,Not Placed
1007,Rahul Singh,Civil,3,82,100,1,Yes,Not Placed
1008,Priya Singh,CSE,6,62,89,1,Yes,Not Placed
1009,Anika Mehta,ECE,2,68,97,0,Yes,Placed
1010,Nisha Chauhan,IT,1,75,74,0,Yes,Placed


In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("student_performance_cleaned")

# format("delta") — saves data in Delta format.
# mode("overwrite") — replaces the table if it already exists.
# saveAsTable() — creates a table you can access later.

In [0]:
display(spark.table("student_performance_cleaned"))

student_id,student_name,department,semester,marks,attendance_percentage,backlogs,fees_paid,placement_status
1004,Aditya Nair,CSE,4,85,82,1,Yes,Not Placed
1025,Neha Singh,ECE,4,68,67,1,Yes,Not Placed
1027,Ananya Chauhan,Civil,2,79,93,1,No,Placed
1044,Aditya Modi,ECE,1,74,80,0,Yes,Placed
1117,Aarav Singh,CSE,1,93,69,0,Yes,Not Placed
1122,Aditya Iyer,CSE,8,80,70,0,Yes,Not Placed
1124,Rahul Rana,CSE,3,81,91,0,Yes,Placed
1126,Ayaan Chauhan,IT,5,83,90,1,Yes,Not Placed
1130,Ayaan Gupta,Civil,4,79,63,0,Yes,Placed
1133,Jiya Gupta,CSE,6,92,88,0,Yes,Placed


In [0]:
%sql
SELECT department, COUNT(*) AS total_students
FROM student_performance_cleaned
GROUP BY department;

department,total_students
CSE,342
ECE,141
Civil,111
IT,250
Mechanical,156


In [0]:
%sql
SELECT department, AVG(marks) AS average_marks
FROM student_performance_cleaned
GROUP BY department
ORDER BY average_marks DESC;

department,average_marks
ECE,76.00709219858156
Mechanical,74.80128205128206
CSE,73.953216374269
IT,73.664
Civil,73.52252252252252


In [0]:
print("Original records:", df.count())
print("Cleaned records:", df_clean.count())
print("Removed records:", df.count() - df_clean.count())

Original records: 1000
Cleaned records: 1000
Removed records: 0
